# Descenso por gradiente

**Capítulo 2 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_optimization/gd.ipynb` · [Lección original](https://d2l.ai/chapter_optimization/gd.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Descenso por gradiente
<a id="sec_gd"></a>

En esta sección vamos a introducir los conceptos básicos subyacentes * descenso por gradiente *. Aunque rara vez se utiliza directamente en el aprendizaje profundo, una comprensión de descenso por gradiente es clave para entender algoritmos de descenso por gradiente estocástico. Por ejemplo, el problema de optimización podría diferir debido a una tasa de aprendizaje demasiado grande. Este fenómeno ya se puede ver en descenso por gradiente. Del mismo modo, el preacondicionamiento es una técnica común en descenso por gradiente y lleva a algoritmos más avanzados. Comencemos con un caso especial simple.

## Descenso por gradiente de una dimensión
El descenso por gradiente en una dimensión es un excelente ejemplo para explicar por qué el algoritmo de descenso por gradiente puede reducir el valor de la función objetiva. Considere algunas funciones de valor real continuamente diferenciables $f: \mathbb{R} \rightarrow \mathbb{R}$.

$$f(x + \epsilon) = f(x) + \epsilon f'(x) + \mathcal{O}(\epsilon^2).$$

:eqlabel:`gd-taylor`

Es decir, en la aproximación de primer orden $f(x+\epsilon)$ es dada por el valor de la función $f(x)$ y la primera derivada $f'(x)$ en $x$. No es irrazonable asumir que para el pequeño $\epsilon$ que se mueve en la dirección del gradiente negativo disminuirá $f$. Para mantener las cosas simples elegimos un paso fijo tamaño $\eta > 0$ y elegir $\epsilon = -\eta f'(x)$.

$$f(x - \eta f'(x)) = f(x) - \eta f'^2(x) + \mathcal{O}(\eta^2 f'^2(x)).$$

:eqlabel:`gd-taylor-2`

Si el derivado $f'(x) \neq 0$ no desaparece, avanzamos desde $\eta f'^2(x)>0$. Además, siempre podemos elegir $\eta$ lo suficientemente pequeño como para que los términos de orden superior se vuelvan irrelevantes.

$$f(x - \eta f'(x)) \lessapprox f(x).$$

Esto significa que, si usamos

$$x \leftarrow x - \eta f'(x)$$

Para iterar $x$, el valor de la función $f(x)$ podría declinar. Por lo tanto, en descenso por gradiente primero elegimos un valor inicial $x$ y un constante $\eta > 0$ y luego los usamos para iterar continuamente $x$ hasta que la condición de parada se alcanza, por ejemplo, cuando la magnitud del gradiente $|f'(x)|$ es suficientemente pequeña o el número de iteraciones ha alcanzado un determinado valor.

Para la simplicidad elegimos la función objetivo $f(x)=x^2$ para ilustrar cómo implementar el descenso por gradiente. Aunque sabemos que $x=0$ es la solución para minimizar $f(x)$, todavía usamos esta función simple para observar cómo cambia $x$.


In [ ]:
%matplotlib inline
import numpy as np
import torch
from laboratorio import d2l

In [ ]:
def f(x):  # Función objetiva
    return x ** 2

def f_grad(x):  # Gradiente (derivado) de la función objetiva
    return 2 * x

A continuación, utilizamos $x=10$ como valor inicial y asumimos $\eta=0.2$. Usando el descenso por gradiente para iterar $x$ durante 10 veces podemos ver que, eventualmente, el valor de $x$ se aproxima a la solución óptima.


In [ ]:
def gd(eta, f_grad):
    x = 10.0
    results = [x]
    for i in range(10):
        x -= eta * f_grad(x)
        results.append(float(x))
    print(f'epoch 10, x: {x:f}')
    return results

results = gd(0.2, f_grad)

El progreso de la optimización sobre $x$ se puede trazar de la siguiente manera.


In [ ]:
def show_trace(results, f):
    n = max(abs(min(results)), abs(max(results)))
    f_line = torch.arange(-n, n, 0.01)
    d2l.set_figsize()
    d2l.plot([f_line, results], [[f(x) for x in f_line], [
        f(x) for x in results]], 'x', 'f(x)', fmts=['-', '-o'])

show_trace(results, f)

### Tasa de aprendizaje
<a id="subsec_gd-learningrate"></a>

La tasa de aprendizaje $\eta$ puede ser establecida por el diseñador de algoritmos. Si usamos una tasa de aprendizaje demasiado pequeña, hará que $x$ se actualice muy lentamente, requiriendo más iteraciones para obtener una mejor solución. Para mostrar lo que sucede en tal caso, considere el progreso en el mismo problema de optimización para $\eta = 0.05$. Como podemos ver, incluso después de 10 pasos todavía estamos muy lejos de la solución óptima.


In [ ]:
show_trace(gd(0.05, f_grad), f)

Por el contrario, si utilizamos una tasa de aprendizaje excesivamente alta, $\left|\eta f'(x)\right|$ podría ser demasiado grande para la fórmula de expansión de Taylor de primer orden. Es decir, el término $\mathcal{O}(\eta^2 f'^2(x))$ en [Referencia gd-taylor-2](https://d2l.ai/#gd-taylor-2) podría llegar a ser significativo. En este caso, no podemos garantizar que la iteración de $x$ será capaz de reducir el valor de $f(x)$. Por ejemplo, cuando establecemos la tasa de aprendizaje a $\eta=1.1$, $x$ supera la solución óptima $x=0$ y diverge gradualmente.


In [ ]:
show_trace(gd(1.1, f_grad), f)

### Mínimos locales
Para ilustrar lo que sucede para las funciones no convexas considere el caso de $f(x) = x \cdot \cos(cx)$ para alguna $c$ constante. Esta función tiene infinitamente muchos mínimos locales. Dependiendo de nuestra elección de la tasa de aprendizaje y dependiendo de lo bien condicionado que es el problema, podemos terminar con una de muchas soluciones. El ejemplo a continuación ilustra cómo una (inrealísticamente) alta tasa de aprendizaje llevará a un mínimo local pobre.


### Nota docente de Hespérides

Sigue tres objetos diferentes: el valor de la pérdida, su gradiente y la actualización que calcula el optimizador. Comprueba formas y reinicia los gradientes antes de cada paso. En el explorador se mantienen función y punto inicial para comparar trayectorias; una misma tasa no significa el mismo desplazamiento efectivo para todos los métodos.

Vínculo con los apuntes: sesión 2, «Descenso por gradiente».


In [ ]:
c = torch.tensor(0.15 * np.pi)

def f(x):  # Función objetiva
    return x * torch.cos(c * x)

def f_grad(x):  # Gradiente de la función objetiva
    return torch.cos(c * x) - c * x * torch.sin(c * x)

show_trace(gd(2, f_grad), f)

## Descenso por gradiente multivariado
Ahora que tenemos una mejor intuición del caso univariado, consideremos la situación donde $\mathbf{x} = [x_1, x_2, \ldots, x_d]^\top$Es decir, la función objetiva $f: \mathbb{R}^d \to \mathbb{R}$ mapas vectores en escalares. Correspondientemente su gradiente es multivariable, también. Es un vector que consiste en $d$ derivados parciales:

$$\nabla f(\mathbf{x}) = \bigg[\frac{\partial f(\mathbf{x})}{\partial x_1}, \frac{\partial f(\mathbf{x})}{\partial x_2}, \ldots, \frac{\partial f(\mathbf{x})}{\partial x_d}\bigg]^\top.$$

Cada elemento derivado parcial $\partial f(\mathbf{x})/\partial x_i$ en el gradiente indica la tasa de cambio de $f$ en $\mathbf{x}$ con respecto a la entrada $x_i$. Como antes en el caso univariado podemos utilizar la aproximación correspondiente de Taylor para funciones multivariadas para tener alguna idea de lo que debemos hacer. En particular, tenemos que

$$f(\mathbf{x} + \boldsymbol{\epsilon}) = f(\mathbf{x}) + \mathbf{\boldsymbol{\epsilon}}^\top \nabla f(\mathbf{x}) + \mathcal{O}(\|\boldsymbol{\epsilon}\|^2).$$

:eqlabel:`gd-multi-taylor`

En otras palabras, hasta los términos de segundo orden en $\boldsymbol{\epsilon}$ la dirección de descenso más pronunciado es dada por el gradiente negativo $-\nabla f(\mathbf{x})$. La elección de una tasa de aprendizaje adecuada $\eta > 0$ produce el algoritmo de descenso por gradiente prototípico:

$$\mathbf{x} \leftarrow \mathbf{x} - \eta \nabla f(\mathbf{x}).$$

Para ver cómo se comporta el algoritmo en la práctica vamos a construir una función objetiva $f(\mathbf{x})=x_1^2+2x_2^2$ con un vector bidimensional $\mathbf{x} = [x_1, x_2]^\top$ como entrada y un escalar como salida. El gradiente es dado por $\nabla f(\mathbf{x}) = [2x_1, 4x_2]^\top$. Observaremos la trayectoria de $\mathbf{x}$ por descenso por gradiente desde la posición inicial $[-5, -2]$.

Para empezar, necesitamos dos funciones de ayuda más. La primera utiliza una función de actualización y la aplica 20 veces al valor inicial. La segunda ayuda visualiza la trayectoria de $\mathbf{x}$.


In [ ]:
def train_2d(trainer, steps=20, f_grad=None):  #@save
    """Optimice una función de objetivo 2D con un entrenador personalizado."""
    # `s1` y `s2` son variables de estado interno que se utilizarán en Momentum, adagrad, RMSProp
    x1, x2, s1, s2 = -5, -2, 0, 0
    results = [(x1, x2)]
    for i in range(steps):
        if f_grad:
            x1, x2, s1, s2 = trainer(x1, x2, s1, s2, f_grad)
        else:
            x1, x2, s1, s2 = trainer(x1, x2, s1, s2)
        results.append((x1, x2))
    print(f'epoch {i + 1}, x1: {float(x1):f}, x2: {float(x2):f}')
    return results

In [ ]:
def show_trace_2d(f, results):  #@save
    """Mostrar el rastro de variables 2D durante la optimización."""
    d2l.set_figsize()
    d2l.plt.plot(*zip(*results), '-o', color='#ff7f0e')
    x1, x2 = torch.meshgrid(torch.arange(-5.5, 1.0, 0.1),
                          torch.arange(-3.0, 1.0, 0.1), indexing='ij')
    d2l.plt.contour(x1, x2, f(x1, x2), colors='#1f77b4')
    d2l.plt.xlabel('x1')
    d2l.plt.ylabel('x2')

A continuación, observamos la trayectoria de la variable de optimización $\mathbf{x}$ para la tasa de aprendizaje $\eta = 0.1$. Podemos ver que después de 20 pasos el valor de $\mathbf{x}$ se aproxima a su mínimo en $[0, 0]$. El progreso es bastante bien comportado, aunque bastante lento.


In [ ]:
def f_2d(x1, x2):  # Función objetiva
    return x1 ** 2 + 2 * x2 ** 2

def f_2d_grad(x1, x2):  # Gradiente de la función objetiva
    return (2 * x1, 4 * x2)

def gd_2d(x1, x2, s1, s2, f_grad):
    g1, g2 = f_grad(x1, x2)
    return (x1 - eta * g1, x2 - eta * g2, 0, 0)

eta = 0.1
show_trace_2d(f_2d, train_2d(gd_2d, f_grad=f_2d_grad))

## Métodos de adaptación
Como podríamos ver en [Referencia subsec_gd-learningrate](https://d2l.ai/chapter_optimization/gd.html#subsec-gd-learningrate), obtener la tasa de aprendizaje $\eta$ "justo derecho" es difícil. Si lo elegimos demasiado pequeño, hacemos poco progreso. Si elegimos demasiado grande, la solución oscila y en el peor de los casos incluso podría divergir. ¿Qué pasa si pudiéramos determinar $\eta$ automáticamente o deshacerse de tener que seleccionar una tasa de aprendizaje en absoluto? Métodos de segundo orden que miran no sólo el valor y el gradiente de la función objetiva, sino también en su *curvatura* puede ayudar en este caso. Si bien estos métodos no se pueden aplicar al aprendizaje profundo directamente debido al costo computacional, proporcionan una intuición útil en cómo diseñar algoritmos de optimización avanzada que imitan muchas de las propiedades deseables de los algoritmos descritos a continuación.

### Método de Newton
Revisando la expansión Taylor de alguna función $f: \mathbb{R}^d \rightarrow \mathbb{R}$ no hay necesidad de parar después del primer término. De hecho, podemos escribir como

$$f(\mathbf{x} + \boldsymbol{\epsilon}) = f(\mathbf{x}) + \boldsymbol{\epsilon}^\top \nabla f(\mathbf{x}) + \frac{1}{2} \boldsymbol{\epsilon}^\top \nabla^2 f(\mathbf{x}) \boldsymbol{\epsilon} + \mathcal{O}(\|\boldsymbol{\epsilon}\|^3).$$

:eqlabel:`gd-hot-taylor`

Para evitar notación engorrosa definimos $\mathbf{H} \stackrel{\textrm{def}}{=} \nabla^2 f(\mathbf{x})$ para ser el Hessian de $f$, que es una matriz $d \times d$. Para pequeños $d$ y problemas simples $\mathbf{H}$ es fácil de calcular. Para redes neuronales profundas, por otro lado, $\mathbf{H}$ puede ser prohibitivamente grande, debido al costo de almacenar entradas $\mathcal{O}(d^2)$. Además, puede ser demasiado caro calcular a través de backpropagation. Por ahora vamos a ignorar tales consideraciones y mirar qué algoritmo que obtendríamos.

Después de todo, el mínimo de $f$ satisface $\nabla f = 0$. Siguiendo las reglas de cálculo en [Referencia subsec_calculus-grad](https://d2l.ai/chapter_preliminaries/calculus.html#subsec-calculus-grad), tomando derivados de [Referencia gd-hot-taylor](https://d2l.ai/#gd-hot-taylor) con respecto a $\boldsymbol{\epsilon}$ e ignorando términos de orden superior llegamos a

$$\nabla f(\mathbf{x}) + \mathbf{H} \boldsymbol{\epsilon} = 0 \textrm{ and hence }
\boldsymbol{\epsilon} = -\mathbf{H}^{-1} \nabla f(\mathbf{x}).$$

Es decir, necesitamos invertir el $\mathbf{H}$ Hessiano como parte del problema de optimización.

Como un simple ejemplo, para $f(x) = \frac{1}{2} x^2$ tenemos $\nabla f(x) = x$ y $\mathbf{H} = 1$. Por lo tanto, para cualquier $x$ obtenemos $\epsilon = -x$. En otras palabras, un paso *single* es suficiente para converger perfectamente sin la necesidad de ningún ajuste! Por desgracia, tenemos un poco de suerte aquí: la expansión de Taylor era exacta desde $f(x+\epsilon)= \frac{1}{2} x^2 + \epsilon x + \frac{1}{2} \epsilon^2$.

Veamos qué sucede en otros problemas. Dada una función de coseno hiperbólica convexa $f(x) = \cosh(cx)$ para alguna constante $c$, podemos ver que el mínimo global en $x=0$ se alcanza después de unas pocas iteraciones.


In [ ]:
c = torch.tensor(0.5)

def f(x):  # Función objetiva
    return torch.cosh(c * x)

def f_grad(x):  # Gradiente de la función objetiva
    return c * torch.sinh(c * x)

def f_hess(x):  # Hessian de la función objetiva
    return c**2 * torch.cosh(c * x)

def newton(eta=1):
    x = 10.0
    results = [x]
    for i in range(10):
        x -= eta * f_grad(x) / f_hess(x)
        results.append(float(x))
    print('epoch 10, x:', x)
    return results

show_trace(newton(), f)

Ahora consideremos una función *nonconvex*, tal como $f(x) = x \cos(c x)$ para alguna constante $c$. Después de todo, notemos que en el método de Newton terminamos dividiendo por el Hessian. Esto significa que si la segunda derivada es *negativa* podemos caminar en la dirección de *aumentar* el valor de $f$.


In [ ]:
c = torch.tensor(0.15 * np.pi)

def f(x):  # Función objetiva
    return x * torch.cos(c * x)

def f_grad(x):  # Gradiente de la función objetiva
    return torch.cos(c * x) - c * x * torch.sin(c * x)

def f_hess(x):  # Hessian de la función objetiva
    return - 2 * c * torch.sin(c * x) - x * c**2 * torch.cos(c * x)

show_trace(newton(), f)

Esto salió espectacularmente mal. ¿Cómo podemos arreglarlo? Una manera sería "arreglar" el Hessian tomando su valor absoluto en su lugar. Otra estrategia es traer de vuelta la tasa de aprendizaje. Esto parece derrotar el propósito, pero no del todo. Tener información de segundo orden nos permite ser cautelosos cuando la curvatura es grande y tomar pasos más largos cuando la función objetiva es más plana. Veamos cómo funciona esto con una tasa de aprendizaje ligeramente menor, digamos $\eta = 0.5$. Como podemos ver, tenemos un algoritmo bastante eficiente.


In [ ]:
show_trace(newton(0.5), f)

### Análisis de convergencia
Sólo analizamos la tasa de convergencia del método de Newton para algunos convexos y tres veces diferenciable función objetivo $f$, donde la segunda derivada es no cero, es decir, $f'' > 0$. La prueba multivariable es una simple extensión del argumento unidimensional a continuación y omitida ya que no nos ayuda mucho en términos de intuición.

Denotar por $x^{(k)}$ el valor de $x$ en la iteración $k^\textrm{th}$ y dejar que $e^{(k)} \stackrel{\textrm{def}}{=} x^{(k)} - x^*$ sea la distancia de la optimidad en la iteración $k^\textrm{th}$. Por expansión Taylor tenemos que la condición $f'(x^*) = 0$ se puede escribir como

$$0 = f'(x^{(k)} - e^{(k)}) = f'(x^{(k)}) - e^{(k)} f''(x^{(k)}) + \frac{1}{2} (e^{(k)})^2 f'''(\xi^{(k)}),$$

que se mantiene para algunos $\xi^{(k)} \in [x^{(k)} - e^{(k)}, x^{(k)}]$. Dividir la expansión anterior por los rendimientos $f''(x^{(k)})$

$$e^{(k)} - \frac{f'(x^{(k)})}{f''(x^{(k)})} = \frac{1}{2} (e^{(k)})^2 \frac{f'''(\xi^{(k)})}{f''(x^{(k)})}.$$

Recordemos que tenemos la actualización $x^{(k+1)} = x^{(k)} - f'(x^{(k)}) / f''(x^{(k)})$. Conectando en esta ecuación de actualización y tomando el valor absoluto de ambos lados, tenemos

$$\left|e^{(k+1)}\right| = \frac{1}{2}(e^{(k)})^2 \frac{\left|f'''(\xi^{(k)})\right|}{f''(x^{(k)})}.$$

En consecuencia, cuando estamos en una región de $\left|f'''(\xi^{(k)})\right| / (2f''(x^{(k)})) \leq c$ limitado, tenemos un error cuadráticamente decreciente

$$\left|e^{(k+1)}\right| \leq c (e^{(k)})^2.$$

Como un lado, los investigadores de optimización llaman a esta convergencia *lineal*, mientras que una condición como $\left|e^{(k+1)}\right| \leq \alpha \left|e^{(k)}\right|$ se llamaría una tasa *constante* de convergencia. Tenga en cuenta que este análisis viene con un número de advertencias. Primero, realmente no tenemos mucha garantía cuando llegaremos a la región de convergencia rápida. En lugar de ello, sólo sabemos que una vez que lo alcancemos, la convergencia será muy rápida. Segundo, este análisis requiere que $f$ se porte bien hasta derivados de orden superior. Se reduce a asegurar que $f$ no tenga ninguna propiedad "sorprendente" en términos de cómo podría cambiar sus valores.

### Preacondicionamiento
No es sorprendente computar y almacenar el Hessian completo es muy caro. Por lo tanto es deseable encontrar alternativas. Una manera de mejorar las cosas es *preacondicionamiento*. Evita computar el Hessian en su totalidad, pero sólo calcula las entradas *diagonales*. Esto conduce a actualizar algoritmos del formulario

$$\mathbf{x} \leftarrow \mathbf{x} - \eta \textrm{diag}(\mathbf{H})^{-1} \nabla f(\mathbf{x}).$$

Si bien esto no es tan bueno como el método de Newton completo, todavía es mucho mejor que no usarlo. Para ver por qué esto podría ser una buena idea considerar una situación en la que una variable denota altura en milímetros y la otra denota altura en kilómetros. Suponiendo que para ambos la escala natural es en metros, tenemos un terrible desajuste en parametrizaciones. Afortunadamente, el uso de preacondicionamiento elimina esto. Preacondicionar efectivamente con descenso por gradiente equivale a seleccionar una tasa de aprendizaje diferente para cada variable (coordinación de vector $\mathbf{x}$). Como veremos más adelante, el preacondicionamiento impulsa algunas de las innovaciones en algoritmos de optimización de descenso por gradiente estocástico.

### Descenso por gradiente con búsqueda de paso
Uno de los problemas clave en el descenso por gradiente es que podemos superar el objetivo o hacer un progreso insuficiente. Una solución simple para el problema es utilizar la búsqueda de línea en conjunción con el descenso por gradiente. Es decir, utilizamos la dirección dada por $\nabla f(\mathbf{x})$ y luego realizar búsquedas binarias en cuanto a qué tasa de aprendizaje $\eta$ minimiza $f(\mathbf{x} - \eta \nabla f(\mathbf{x}))$.

Este algoritmo converge rápidamente (para un análisis y una prueba ver, por ejemplo, [Boyd.Vandenberghe.2004](https://d2l.ai/chapter_references/zreferences.html)). Sin embargo, para el propósito de aprendizaje profundo esto no es tan factible, ya que cada paso de la búsqueda de la línea requeriría evaluar la función objetiva en todo el conjunto de datos. Esto es demasiado costoso de lograr.

## Resumen
* Los índices de aprendizaje importan. Demasiado grandes y divergimos, demasiado pequeños y no avanzamos.
* El descenso por gradiente puede quedar atascado en los mínimos locales.
* En las grandes dimensiones, el ajuste de la tasa de aprendizaje es complicado.
* El preacondicionamiento puede ayudar con el ajuste de la escala.
* El método de Newton es mucho más rápido una vez que ha comenzado a trabajar correctamente en problemas convexos.
* Tenga cuidado de usar el método de Newton sin ningún ajuste para problemas no convexos.

## Ejercicios
1. Experimenta con diferentes tasas de aprendizaje y funciones objetivas para descenso por gradiente.
1. Implementar la búsqueda de paso para minimizar una función convexa en el intervalo $[a, b]$.
    1. ¿Necesitas derivadas para la búsqueda binaria, es decir, para decidir si elegir $[a, (a+b)/2]$ o $[(a+b)/2, b]$.
    1. ¿Qué tan rápido es la tasa de convergencia para el algoritmo?
    1. Implementar el algoritmo y aplicarlo para minimizar $\log (\exp(x) + \exp(-2x -3))$.
1. Diseñar una función objetiva definida en $\mathbb{R}^2$ donde el descenso por gradiente es extremadamente lento.
1. Implementar la versión ligera del método de Newton usando preacondicionamiento:
    1. Utilice Hessian diagonal como precondicional.
    1. Utilice los valores absolutos de eso en lugar de los valores reales (posiblemente firmados).
    1. Aplicar esto al problema anterior.
1. Aplicar el algoritmo anterior a un número de funciones objetivas (convexas o no). ¿Qué sucede si rota coordenadas por $45$ grados?

[Debate del original](https://discuss.d2l.ai/t/351)
